In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 transformers peft accelerate datasets

In [ ]:
import os, re, gc
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"PyTorch: {torch.__version__}")
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1e9:.1f} GB")

PyTorch: 2.10.0+cu128
GPUs available: 1
  GPU 0: Tesla T4, 15.6 GB


In [ ]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
kagglehub.competition_download('llm-classification-finetuning')

100%|██████████| 57.0M/57.0M [00:00<00:00, 77.2MB/s]

Extracting files...


'/root/.cache/kagglehub/competitions/llm-classification-finetuning'

## Config

In [ ]:
# ─── MODEL
MODEL_ID   = "meta-llama/Llama-3.2-1B"

DEVICE_MAP = "cuda:0"

# ─── SPEED / MEMORY
MAX_LEN    = 512
BATCH_SIZE = 32
GRAD_ACCUM = 1
LR         = 2e-4
EPOCHS     = 1
SEED       = 42

# ─── LORA
LORA_R     = 16
LORA_ALPHA = 32

DATA_DIR   = "/root/.cache/kagglehub/competitions/llm-classification-finetuning"

## Load full dataset

In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(f"Train: {len(train_df):,} rows | Test: {len(test_df):,} rows")
print(train_df[['winner_model_a','winner_model_b','winner_tie']].sum())

Train: 57,477 rows | Test: 3 rows
winner_model_a    20064
winner_model_b    19652
winner_tie        17761
dtype: int64


## 🧹 Preprocessing
**Key insight from top solutions:** Don't truncate from the front.
Keep the beginning (where structure/quality is established) and the end
(where conclusions live). Drop the repetitive middle.
This + markdown stripping fits more signal into 512 tokens.

In [ ]:
def clean_text(text: str) -> str:
    """Strip markdown noise, keep semantic content."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"```[\w]*\n?", "", text)        # code fence markers
    text = re.sub(r"\*{1,3}(.*?)\*{1,3}", r"\1", text)  # bold/italic
    text = re.sub(r"^#{1,6}\s+", "", text, flags=re.MULTILINE)  # headers
    text = re.sub(r"\n{3,}", "\n\n", text)          # excess newlines
    return text.strip()


def smart_truncate(text: str, max_chars: int = 800) -> str:
    """Keep start + end, drop middle. Avoids losing conclusions."""
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    return text[:half] + " [...] " + text[-half:]


def format_text(row) -> str:
    p = smart_truncate(clean_text(str(row['prompt'])),   600)
    a = smart_truncate(clean_text(str(row['response_a'])), 800)
    b = smart_truncate(clean_text(str(row['response_b'])), 800)
    return (
        f"Prompt: {p}\n\n"
        f"Response A: {a}\n\n"
        f"Response B: {b}"
    )


def get_label(row) -> int:
    if row['winner_model_a'] == 1: return 0
    if row['winner_model_b'] == 1: return 1
    return 2


train_df['text']  = train_df.apply(format_text, axis=1)
test_df['text']   = test_df.apply(format_text, axis=1)
train_df['label'] = train_df.apply(get_label, axis=1)

print("Label distribution:")
print(train_df['label'].value_counts())

# Check token length distribution to validate MAX_LEN choice
sample_lengths = train_df['text'].sample(500).str.len()
print(f"\nText char length — mean: {sample_lengths.mean():.0f}, "
      f"p90: {sample_lengths.quantile(0.9):.0f}, max: {sample_lengths.max()}")

Label distribution:
label
0    20064
1    19652
2    17761
Name: count, dtype: int64

Text char length — mean: 1410, p90: 1982, max: 2257


## Model setup

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"  # required for causal LM + classification head

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=3,
    quantization_config=bnb_config,
    device_map=DEVICE_MAP,
    dtype=torch.float16,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False  # required for gradient checkpointing with Llama

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["score"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 8,800,896 || all params: 502,836,352 || trainable%: 1.7503


## Tokenize with dynamic padding

In [ ]:
train_ds = Dataset.from_pandas(train_df[['text', 'label']])
test_ds  = Dataset.from_pandas(test_df[['text']])

def tokenize_func(examples):
    return tokenizer(
        examples["text"],
        padding=False,
        truncation=True,
        max_length=MAX_LEN,
    )

train_tokenized = train_ds.map(
    tokenize_func, batched=True,
    remove_columns=["text"],
    num_proc=2,                  # parallel tokenization
)
test_tokenized = test_ds.map(
    tokenize_func, batched=True,
    remove_columns=["text"],
    num_proc=2,
)

# Dynamic padding collator — pads each batch to its own max, pad to multiple of 8
# for tensor core alignment (small but free speed gain)
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

print(f"Train samples: {len(train_tokenized):,}")
print(f"Test  samples: {len(test_tokenized):,}")

Map (num_proc=2):   0%|          | 0/57477 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/3 [00:00<?, ? examples/s]

Train samples: 57,477
Test  samples: 3


## Training

In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    bf16=False,
    gradient_checkpointing=False,
    optim="paged_adamw_8bit",
    label_smoothing_factor=0.1,             # improves MAP score
    # group_by_length=True,                   # batch similar lengths → less padding waste
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to="none",
    logging_steps=50,
    save_strategy="no",
    seed=SEED,
    # no_cuda=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    data_collator=data_collator,
)

trainer._wrap_model = lambda model, training=True, dataloader=None: model

print("Training...")
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training...


`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,3.680860
100,1.438034
150,1.329274
200,1.180689
250,1.189061
300,1.137464
350,1.128466
400,1.122844
450,1.113813
500,1.117575


TrainOutput(global_step=1797, training_loss=1.1636883263861264, metrics={'train_runtime': 11629.1386, 'train_samples_per_second': 4.942, 'train_steps_per_second': 0.155, 'total_flos': 6.461535000841421e+16, 'train_loss': 1.1636883263861264, 'epoch': 1.0})

## Inference & Submission

In [ ]:
# Free training memory before inference
gc.collect()
torch.cuda.empty_cache()

print("Generating predictions...")
preds = trainer.predict(test_tokenized).predictions

probs = torch.nn.functional.softmax(
    torch.tensor(preds, dtype=torch.float32), dim=-1
).numpy()

submission = pd.DataFrame({
    'id':             test_df['id'],
    'winner_model_a': probs[:, 0],
    'winner_model_b': probs[:, 1],
    'winner_tie':     probs[:, 2],
})

submission.to_csv('submission.csv', index=False)
print("Done! submission.csv written.")
print(submission.head())

# Sanity: each row must sum to 1.0
row_sums = submission[['winner_model_a','winner_model_b','winner_tie']].sum(axis=1)
print(f"\nRow sum check — mean: {row_sums.mean():.4f}, std: {row_sums.std():.6f}")

Generating predictions...


Done! submission.csv written.
        id  winner_model_a  winner_model_b  winner_tie
0   136060        0.233441        0.388894    0.377665
1   211333        0.388344        0.330711    0.280945
2  1233961        0.361443        0.329097    0.309460

Row sum check — mean: 1.0000, std: 0.000000


In [ ]:
!zip -r /content/my_model.zip /kaggle/working

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/results/ (stored 0%)


In [ ]:
from google.colab import files
files.download('/content/my_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>